# Employee Retention Workflow

## Purpose

This notebook documents the data preparation workflow for the Employee Retention Dashboard.

The existing `workforce_dashboard.csv` is assignment-level data. It is useful for workforce and assignment analysis, but it should not be used by itself to calculate employee retention or turnover.

For employee retention, we use employee-level employment and offboarding data from MariaDB.


## 1. Verify the source tables

We first checked the number of records in the employee-related tables.


In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS employees FROM employees;
SELECT COUNT(*) AS employment_history FROM employment_history;
SELECT COUNT(*) AS offboarding FROM offboarding;
"

### Verified record counts

- Employees: **1,600**
- Employment history: **1,600**
- Offboarding: **320**

These counts were verified directly against MariaDB.


## 2. Inspect the retention-related fields

The employment history table contains employment dates, status, position, career level, start reason, and end reason.

The offboarding table contains departure information, leaving reason, exit satisfaction, recommendation/return responses, and rehire eligibility.


In [ ]:
sudo mariadb -e "
USE daimonupdown;
DESCRIBE employment_history;
DESCRIBE offboarding;
"

## 3. Create the employee retention dataset

We joined employees to employment history and then joined offboarding using the employment history ID.

A LEFT JOIN was used so employees without an offboarding record remain in the dataset. This is important because currently active employees may not have offboarding information.


In [ ]:
sudo mariadb --batch -e "
USE daimonupdown;
SELECT
    e.employee_id,
    e.first_name,
    e.last_name,
    eh.employment_start_date,
    eh.employment_end_date,
    eh.employment_status,
    eh.employment_type,
    eh.position,
    eh.career_level,
    eh.start_reason,
    eh.end_reason,
    o.offboarding_date,
    o.departure_type,
    o.leaving_reason,
    o.exit_satisfaction,
    o.would_recommend,
    o.would_return,
    o.rehire_eligible
FROM employees e
LEFT JOIN employment_history eh
    ON e.employee_id = eh.employee_id
LEFT JOIN offboarding o
    ON eh.employment_history_id = o.employment_history_id;
" > dashboard/data/employee_retention.tsv

## 4. Convert TSV to CSV

The SQL export was saved as a TSV file so it could be converted into a CSV for Tableau.

The existing `tsv_to_csv.py` script initially had an incorrect delimiter value. The delimiter was corrected from `"\\\\t"` to the actual tab character `"\\t"`.


In [ ]:
python python/tsv_to_csv.py dashboard/data/employee_retention.tsv

The conversion completed successfully and created:

`dashboard/data/employee_retention.csv`


In [ ]:
head -n 2 dashboard/data/employee_retention.csv

## 5. Verified CSV structure

The CSV contains employee, employment, and offboarding fields needed for the retention analysis.

Example verified fields include:

- `employee_id`
- `employment_start_date`
- `employment_end_date`
- `employment_status`
- `employment_type`
- `position`
- `career_level`
- `start_reason`
- `end_reason`
- `offboarding_date`
- `departure_type`
- `leaving_reason`
- `exit_satisfaction`
- `would_recommend`
- `would_return`
- `rehire_eligible`

The first verified employee record had an `Active` employment status and no offboarding record, demonstrating why the LEFT JOIN is useful.


## 6. Retention analysis direction

The retention dashboard will focus on employee-level questions:

1. How many employees are currently active?
2. How many employees have been offboarded?
3. What is the employee retention/turnover picture?
4. How often do employees leave over time?
5. What are the main reasons employees leave?
6. How does retention or turnover vary by position or other available employee attributes?

Important: an overall percentage based only on the current counts should not automatically be labeled an annual retention rate. A time period must be defined before calculating an annual retention or turnover rate.


## 7. Next step

Load `dashboard/data/employee_retention.csv` into Tableau and build the Employee Retention Dashboard from the verified employee-level data.
